(ollama2)=
# Introducción a Ollama
```{index} Ollama
```

Ollama es una herramienta revolucionaria diseñada para simplificar la descarga, instalación y ejecución de grandes modelos de lenguaje (LLMs) de forma local. Tradicionalmente, trabajar con modelos como Llama 3, Mistral o Gemma requería configurar entornos complejos de Python, manejar dependencias de GPU (como CUDA) y descargar manualmente gigabytes de pesos de modelos. Ollama abstrae toda esta complejidad y la empaqueta en una solución ligera y eficiente.

## ¿Por qué ejecutar LLMs locales en Windows?

El uso de LLMs de forma local en entornos Windows ofrece ventajas estratégicas cruciales:
* **Privacidad y Seguridad de Datos**: La información procesada nunca sale de tu máquina local. Esto es vital para entornos corporativos o proyectos con datos confidenciales.
* **Sin Costes por Token**: A diferencia de las APIs de OpenAI o Anthropic, el uso de Ollama es completamente gratuito una vez descargado el modelo, limitado únicamente por los recursos de hardware de tu sistema.
* **Baja Latencia**: Al eliminar la necesidad de comunicación por red con servidores externos, las respuestas pueden generarse con latencias mínimas, siempre que el hardware local sea el adecuado.
* **Desarrollo sin Conexión**: Puedes seguir programando y experimentando con IA incluso sin acceso a internet.

---

## Requisitos del Sistema e Instalación en Windows

Ollama cuenta con soporte nativo para Windows, lo que significa que no necesitas configurar el Subsistema de Windows para Linux (WSL2) de manera obligatoria, aunque sigue siendo compatible.

### Requisitos mínimos y recomendados

* **Sistema Operativo**: Windows 10 (versión 19041 o superior) o Windows 11.
* **Memoria RAM**: 
  * Para modelos de 7B-8B parámetros (ej. Llama 3): Mínimo 8 GB, recomendado 16 GB.
  * Para modelos de 13B-14B parámetros (ej. Mistral Small): Mínimo 16 GB, recomendado 32 GB.
* **GPU (Opcional pero altamente recomendado)**: Tarjeta gráfica NVIDIA (GeForce RTX o superior) o AMD con soporte para aceleración por hardware. Ollama detectará automáticamente tu GPU para acelerar la inferencia. Si no dispones de una, se utilizará la CPU, lo que resultará en velocidades de respuesta notablemente más lentas.

### Proceso de Instalación

1. **Descargar el Instalador**: Dirígete al sitio web oficial de Ollama y descarga el ejecutable para Windows (`OllamaSetup.exe`).
2. **Ejecutar el Instalador**: Abre el archivo descargado y sigue los pasos del asistente. El proceso instalará Ollama en tu sistema y lo configurará como un servicio en segundo plano.
3. **Verificación**: Una vez instalado, verás el icono de la llama en la bandeja del sistema (system tray). Abre la consola de comandos de Windows (CMD o PowerShell) y ejecuta el siguiente comando para verificar que está correctamente instalado:
   ```bash
   ollama --version
   ```

---

## Gestión y Operación de Modelos con Ollama

Ollama funciona mediante una interfaz de línea de comandos (CLI) intuitiva. El servicio por defecto escucha en `http://localhost:11434`.

### Comandos Esenciales

* **Descargar y ejecutar un modelo**: El comando `run` descarga el modelo (si no está ya en el sistema) y arranca una sesión de chat interactiva.
  ```powershell
  ollama run llama3
  ```
* **Listar modelos instalados**: Muestra los modelos que tienes disponibles localmente.
  ```powershell
  ollama list
  ```
* **Descargar un modelo sin ejecutarlo**: Ideal para preparar el entorno con antelación.
  ```powershell
  ollama pull mistral
  ```
* **Eliminar un modelo**: Libera espacio en disco borrando el modelo especificado.
  ```powershell
  ollama rm gemma
  ```

### El Archivo de Configuración (Modelfile)

Ollama permite personalizar el comportamiento de los modelos mediante un archivo llamado `Modelfile`. Puedes definir el sistema de prompts (instrucciones base), la temperatura y otros hiperparámetros.

Ejemplo de un archivo `Modelfile` para crear un asistente experto en Python:

```dockerfile
FROM llama3

# Establecemos la temperatura (más baja para mayor precisión)
PARAMETER temperature 0.2

# Definimos el prompt del sistema
SYSTEM """
Eres un ingeniero de software senior experto en Python. Responde de forma concisa, 
utilizando siempre tipado estático y buenas prácticas de desarrollo.
"""
```

Para crear este modelo personalizado, guarda el contenido en un archivo sin extensión llamado `Modelfile` y ejecuta en PowerShell:

```powershell
ollama create python-expert -f ./Modelfile
```

Posteriormente, podrás usar tu modelo como cualquier otro:
```powershell
ollama run python-expert
```

---

## Integración de Ollama con LangChain

LangChain es el framework líder para el desarrollo de aplicaciones basadas en Modelos de Lenguaje. Permite encadenar llamadas a LLMs, gestionar la memoria, conectar datos externos (RAG) y construir agentes complejos.

### Configuración del Entorno de Python

Para conectar Ollama con LangChain en Windows, primero necesitas crear un entorno virtual de Python e instalar las librerías necesarias. Abre PowerShell y ejecuta:

```powershell
# Crear y activar un entorno virtual
python -m venv .venv
.\.venv\Scripts\Activate.ps1

# Instalar langchain y el paquete específico para Ollama
pip install langchain langchain-community langchain-core
```

### Ejemplo 1: Inferencia Básica con `ChatOllama`

En LangChain, la clase `ChatOllama` se utiliza para interactuar con modelos de chat locales servidos por Ollama.

Crea un archivo llamado `basico.py`:

```python
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage

# Inicializar el modelo
# Asegúrate de haber ejecutado previamente 'ollama pull llama3'
llm = ChatOllama(
    model="llama3",
    temperature=0.7
)

# Definir la consulta
mensaje = HumanMessage(content="¿Cuáles son las 3 principales ventajas de usar Python para la IA?")

# Obtener respuesta
print("Enviando consulta a Ollama local...")
respuesta = llm.invoke([mensaje])

print("\nRespuesta del modelo:")
print(respuesta.content)
```

### Ejemplo 2: Creación de Cadenas (Chains) con Prompts Estructurados

La verdadera potencia de LangChain surge al estructurar los prompts y procesar las respuestas mediante el Lenguaje de Expresión de LangChain (LCEL).

Crea un archivo llamado `cadena_traduccion.py`:

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.chat_models import ChatOllama
from langchain_core.output_parsers import StrOutputParser

# 1. Definir la plantilla del prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un traductor experto. Traduce el siguiente texto al {idioma_destino}."),
    ("user", "{texto}")
])

# 2. Inicializar el modelo de Ollama
llm = ChatOllama(model="llama3", temperature=0.3)

# 3. Construir la cadena utilizando LCEL
cadena = prompt | llm | StrOutputParser()

# 4. Ejecutar la cadena
idioma = "alemán"
texto_a_traducir = "Hola, estoy aprendiendo a usar modelos locales de inteligencia artificial en mi ordenador."

print(f"Traduciendo a {idioma}...")
resultado = cadena.invoke({
    "idioma_destino": idioma,
    "texto": texto_a_traducir
})

print("\nTraducción:")
print(resultado)
```

### Ejemplo 3: Respuestas en Streaming

Para mejorar la experiencia de usuario, puedes configurar LangChain para que imprima la respuesta del modelo en tiempo real a medida que se genera.

```python
from langchain_community.chat_models import ChatOllama

llm = ChatOllama(model="llama3")

query = "¿Me puedes explicar brevemente qué es la computación cuántica?"

print("Respuesta en tiempo real:\n")
for chunk in llm.stream(query):
    print(chunk.content, end="", flush=True)
print()
```

---

## Arquitectura Avanzada: Implementación de RAG local con Ollama y LangChain

El patrón **RAG (Generación Aumentada por Recuperación)** permite al modelo local responder preguntas basándose en documentos privados (PDFs, TXT, bases de datos) que no estaban en su entrenamiento original.

Para implementar un RAG completamente local en Windows necesitas instalar paquetes adicionales para procesar texto y crear una base de datos vectorial local (ChromaDB):

```powershell
pip install chromadb langchain-text-splitters
```

### Ejemplo completo de RAG Local

A continuación se muestra un script que simula la ingesta de documentos, su almacenamiento en una base de datos vectorial en memoria y la posterior consulta utilizando Ollama para generar la respuesta final.

```python
from langchain_community.chat_models import ChatOllama
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Documentos de prueba (pueden ser cargados desde archivos locales)
documentos = [
    "El proyecto secreto 'X-Apollo' es una iniciativa para migrar todos los servidores a energía solar antes de 2028.",
    "El presupuesto asignado para el proyecto 'X-Apollo' es de 15 millones de euros y está liderado por la ingeniera Elena Torres.",
    "La empresa cerrará sus oficinas físicas en diciembre de 2026 para pasar a un modelo de trabajo 100% remoto."
]

# 2. Dividir el texto en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
fragmentos = text_splitter.create_documents(documentos)

# 3. Crear Embeddings con Ollama
# Asegúrate de haber ejecutado 'ollama pull nomic-embed-text' antes
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# 4. Crear Base de Datos Vectorial (ChromaDB en memoria)
vectorstore = Chroma.from_documents(documents=fragmentos, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# 5. Definir el prompt para RAG
template = """Responde a la pregunta basándote únicamente en el siguiente contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:"""

prompt_rag = ChatPromptTemplate.from_template(template)

# 6. Inicializar el LLM
llm = ChatOllama(model="llama3", temperature=0)

# 7. Construir la cadena RAG
cadena_rag = (
    {"contexto": retriever | (lambda docs: "\n".join([d.page_content for d in docs])), "pregunta": RunnablePassthrough()}
    | prompt_rag
    | llm
    | StrOutputParser()
)

# 8. Realizar una consulta
pregunta = "¿Quién lidera el proyecto X-Apollo y cuál es su presupuesto?"
print(f"Pregunta: {pregunta}\n")

respuesta = cadena_rag.invoke(pregunta)
print("Respuesta generada:")
print(respuesta)
```

---

## Solución de Problemas y Buenas Prácticas en Windows

### Problemas comunes y soluciones

* **Ollama no detecta la GPU NVIDIA**:
  Asegúrate de tener instalados los drivers de NVIDIA más recientes. Puedes comprobar si Windows reconoce correctamente tu tarjeta gráfica ejecutando `nvidia-smi` en la terminal. Si ves información sobre tu GPU y la versión de CUDA instalada, Ollama debería reconocerla automáticamente al iniciarse.
* **Error: `connection refused` en LangChain**:
  Este error ocurre si el servicio de Ollama no se está ejecutando en segundo plano. Comprueba que el icono de Ollama esté en la barra de tareas de Windows. Puedes iniciarlo manualmente buscando "Ollama" en el menú inicio.
* **Rendimiento excesivamente lento**:
  * Asegúrate de no estar ejecutando otros procesos pesados en segundo plano que saturen la memoria RAM o la VRAM.
  * Elige modelos más pequeños. Si el modelo `llama3` (8B) es demasiado pesado para tu sistema, prueba con modelos optimizados como `phi3` de Microsoft o `qwen2` (0.5B o 1.5B), que requieren muchos menos recursos y ofrecen un rendimiento excelente para tareas generales.

### Buenas Prácticas

* **Fijar variables de entorno**: Puedes configurar variables de entorno en Windows para modificar el comportamiento de Ollama. Por ejemplo, `OLLAMA_NUM_PARALLEL=4` permite atender múltiples peticiones en paralelo.
* **Limpieza de caché**: De vez en cuando, revisa el espacio que ocupan los modelos. En Windows, por defecto, los modelos se guardan en `C:\Users\<TuUsuario>\.ollama\models`. Si cambias de disco y quieres ahorrar espacio en el disco del sistema (`C:`), puedes establecer la variable de entorno de sistema `OLLAMA_MODELS` apuntando a otra unidad de almacenamiento.
```